# Comparing XAI methods on one shared tabular case

**Question:** how does a model distinguish malignant from benign samples using cell measurements, and what can different explainers tell us about the same prediction?

We use scikit-learn's bundled Wisconsin Diagnostic Breast Cancer dataset, one stratified split, and one fixed patient row. The reference model is a random forest. No dataset download is needed. These examples describe learned model behavior; feature changes are neither causal effects nor treatment recommendations.

Most sections explain this exact forest. Integrated Gradients needs a differentiable model, so it uses a separately evaluated neural network. RuleFit and a shallow tree are explicitly labeled surrogate models. MERLIN compares a simpler forest with the reference forest on the same cases. Different questions require different outputs: there is no universal numeric score that ranks all these methods.

**Setup:** Python 3.11, `requirements-dev.txt`, `requirements-merlin.txt`, and Java 21 for H2O. The optional MERLIN build instructions are in the repository README. All sections are executable; missing prerequisites produce an error rather than a silently skipped method.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score

np.random.seed(42)
dataset = load_breast_cancer()
feature_names = [name.replace(" ", "_") for name in dataset.feature_names]
X = pd.DataFrame(dataset.data, columns=feature_names)
y = (dataset.target == 0).astype(int)  # 1 = malignant; 0 = benign
class_names = ["benign", "malignant"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)
model = RandomForestClassifier(
    n_estimators=150, max_depth=6, min_samples_leaf=2,
    random_state=42, n_jobs=2
)
model.fit(X_train, y_train)
probabilities = model.predict_proba(X_test)[:, 1]
predictions = model.predict(X_test)
display(pd.DataFrame([{
    "Model": "Reference random forest",
    "Test accuracy": accuracy_score(y_test, predictions),
    "Test balanced accuracy": balanced_accuracy_score(y_test, predictions),
    "Test ROC AUC": roc_auc_score(y_test, probabilities),
    "Training rows": len(X_train), "Test rows": len(X_test),
}]))

## Fix the case before explaining it

For a readable illustration, select a correctly predicted malignant test sample whose forest probability is closest to 0.70. This selection is disclosed and used only for the explanation; the metrics above use the entire held-out set. Every local tabular explanation below uses this same row and class 1.

In [ ]:
candidates = np.flatnonzero((y_test == 1) & (predictions == 1))
assert len(candidates) > 0, "The reference model did not correctly identify any malignant test rows."
query_position = int(candidates[np.argmin(np.abs(probabilities[candidates] - 0.70))])
query = X_test.iloc[[query_position]].copy()
query_values = query.iloc[0].to_numpy()
explained_class = 1
print(f"Dataset row: {query.index[0]}; test position: {query_position}")
print(f"Observed class: {class_names[y_test[query_position]]}")
print(f"Reference model P(malignant): {probabilities[query_position]:.3f}")
display(query.T.rename(columns={query.index[0]: "Shared case"}))

## What is comparable?

| Method | Question | Model used |
|---|---|---|
| Impurity, permutation, drop-column | Which features matter globally, under different definitions? | Reference forest / refitted forest |
| SHAP, LIME | Which features contribute to this prediction? | Reference forest |
| PDP, ICE, ALE | How does the predicted probability vary with a feature? | Reference forest |
| Anchors | Which conditions preserve the local predicted class under sampled perturbations? | Reference forest |
| DiCE | Which alternative observed feature vectors receive the opposite prediction? | Reference forest |
| Shallow surrogate, RuleFit | Can simpler rules approximate the forest? | Surrogates of the reference forest |
| MERLIN | How do rules differ between two model versions? | Simpler forest versus reference forest |
| Integrated Gradients | How does a neural prediction change from a baseline to this case? | Separate tabular neural network |

Grad-CAM requires convolutional feature maps, and attention visualization requires an attention model. LIME/SHAP image and text variants and survival-specific SHAP are different data/model settings, so they are not run on this tabular classification case. The corresponding notebooks remain available in `Tools/`.

## Global feature importance: three different questions

All methods below start from the same fitted random forest and the same train/test split. **MDI** measures how much each feature reduced impurity during training. **Permutation importance** measures the loss in held-out ROC AUC when a feature is shuffled. **Drop-column importance** retrains a new forest without one feature and measures the loss in held-out ROC AUC.

The drop-column calculation is restricted to the five highest-MDI features, selected using the training-fitted model. Its refitted models differ from the original forest. These quantities have different meanings: do not compare MDI magnitudes with AUC losses or average them into a common score. Correlated measurements can substitute for one another, and permutation or drop-column importance can be near zero or negative.

In [ ]:
from sklearn.base import clone
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score

positive_class_index = int(np.flatnonzero(model.classes_ == explained_class)[0])
assert explained_class == 1 and list(model.classes_) == [0, 1]
reference_test_probability = model.predict_proba(X_test)[:, positive_class_index]
reference_auc = roc_auc_score(y_test, reference_test_probability)

mdi_table = pd.DataFrame({
    'feature': feature_names,
    'MDI': model.feature_importances_,
}).sort_values('MDI', ascending=False).reset_index(drop=True)

# Eight shuffles describe sensitivity to the permutation seed, not a confidence interval.
permutation_result = permutation_importance(
    model, X_test, y_test, scoring='roc_auc', n_repeats=8,
    random_state=42, n_jobs=2,
)
permutation_table = pd.DataFrame({
    'feature': feature_names,
    'AUC decrease': permutation_result.importances_mean,
    'shuffle SD': permutation_result.importances_std,
}).sort_values('AUC decrease', ascending=False).reset_index(drop=True)

assert np.isfinite(mdi_table['MDI']).all()
assert np.isfinite(permutation_result.importances).all()
np.testing.assert_allclose(mdi_table['MDI'].sum(), 1.0)
print(f'Original forest held-out ROC AUC: {reference_auc:.3f}')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
mdi_top = mdi_table.head(10).iloc[::-1]
axes[0].barh(mdi_top['feature'], mdi_top['MDI'], color='#3465a4')
axes[0].set(title='Training impurity importance: top 10', xlabel='Mean decrease in impurity')
perm_top = permutation_table.head(10).iloc[::-1]
axes[1].barh(perm_top['feature'], perm_top['AUC decrease'],
            xerr=perm_top['shuffle SD'], color='#d47c22', capsize=3)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set(title='Held-out permutation importance: top 10', xlabel='ROC AUC decrease (mean ± shuffle SD)')
plt.tight_layout()
plt.show()

In [ ]:
drop_column_rows = []
for feature in mdi_table.head(5)['feature']:
    reduced_model = clone(model)
    reduced_model.fit(X_train.drop(columns=feature), y_train)
    reduced_probability = reduced_model.predict_proba(X_test.drop(columns=feature))[:, positive_class_index]
    reduced_auc = roc_auc_score(y_test, reduced_probability)
    drop_column_rows.append({
        'removed feature': feature,
        'refitted model AUC': reduced_auc,
        'AUC decrease after refitting': reference_auc - reduced_auc,
    })
drop_column_table = pd.DataFrame(drop_column_rows)
assert len(drop_column_table) == 5
assert np.isfinite(drop_column_table.select_dtypes(include='number')).all().all()
display(drop_column_table)
print('Each row is a separately refitted model. Negative decreases mean the refitted model scored higher.')

## SHAP: global magnitude and the selected case

Tree SHAP explains the **original random forest's malignant-class probability**. The global chart averages absolute attributions over the held-out cases. The waterfall explains exactly the selected `query`: signed contributions add to the difference between its probability and the explainer's baseline.

Here `TreeExplainer` uses its tree-path-dependent background, based on training observations recorded in the tree paths. This differs from an interventional background and affects how correlated measurements share credit. The numerical check below verifies additivity; it does not establish a causal interpretation.

In [ ]:
import shap

shap_explainer = shap.TreeExplainer(model)
shap_test = shap_explainer(X_test)
shap_query = shap_explainer(query)
assert shap_test.values.shape == (len(X_test), len(feature_names), len(class_names))
assert np.isfinite(shap_test.values).all()
positive_shap_values = shap_test.values[:, :, positive_class_index]
np.testing.assert_allclose(
    shap_test.base_values[:, positive_class_index] + positive_shap_values.sum(axis=1),
    reference_test_probability, atol=1e-6,
)
shap_global_table = pd.DataFrame({
    'feature': feature_names,
    'mean absolute SHAP': np.abs(positive_shap_values).mean(axis=0),
}).sort_values('mean absolute SHAP', ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(8, 4))
shap_top = shap_global_table.head(10).iloc[::-1]
ax.barh(shap_top['feature'], shap_top['mean absolute SHAP'], color='#b04b68')
ax.set(title='Global Tree SHAP: malignant class', xlabel='Mean absolute contribution to probability')
plt.tight_layout()
plt.show()

local_shap = shap_query[0, :, positive_class_index]
assert local_shap.values.shape == (len(feature_names),)
np.testing.assert_allclose(
    local_shap.base_values + local_shap.values.sum(),
    model.predict_proba(query)[0, positive_class_index], atol=1e-6,
)
plt.figure()
shap.plots.waterfall(local_shap, max_display=10, show=False)
plt.title('Same query: SHAP contributions to malignant probability')
plt.tight_layout()
plt.show()

## LIME: a local approximation for the same case

LIME perturbs the selected case and fits a weighted local linear surrogate to the **same forest's malignant probability**. Its bars describe discretized feature conditions, while SHAP's bars describe feature contributions under a different attribution definition. Their magnitudes and baselines are therefore not interchangeable.

The printed weighted local R² measures fit to LIME's sampled neighborhood. Its local prediction can differ from the forest's probability. Perturbations may combine correlated medical measurements in unrealistic ways; this is an explanation of model behavior, not a medical recommendation.

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

# LIME passes NumPy arrays; restore the column names expected by the fitted forest.
def forest_probabilities(rows):
    return model.predict_proba(pd.DataFrame(rows, columns=feature_names))

lime_explainer = LimeTabularExplainer(
    X_train.to_numpy(), feature_names=feature_names,
    class_names=class_names, mode='classification',
    discretize_continuous=True, random_state=42,
)
lime_query = lime_explainer.explain_instance(
    query_values, forest_probabilities, labels=(explained_class,),
    num_features=10, num_samples=3000,
)
lime_table = pd.DataFrame(lime_query.as_list(label=explained_class),
                          columns=['local feature condition', 'LIME weight'])
assert len(lime_table) == 10 and np.isfinite(lime_table['LIME weight']).all()
lime_local_prediction = float(np.asarray(lime_query.local_pred).ravel()[0])
forest_query_probability = float(model.predict_proba(query)[0, positive_class_index])
assert np.isfinite([lime_local_prediction, lime_query.score]).all()
print(f'Original forest P(malignant): {forest_query_probability:.4f}')
print(f'LIME local surrogate prediction: {lime_local_prediction:.4f}')
print(f'Absolute local approximation gap: {abs(lime_local_prediction - forest_query_probability):.4f}')
print(f'LIME weighted local R²: {lime_query.score:.3f}')

lime_plot = lime_table.iloc[::-1]
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(lime_plot['local feature condition'], lime_plot['LIME weight'],
        color=['#b04b68' if value > 0 else '#3465a4' for value in lime_plot['LIME weight']])
ax.axvline(0, color='black', linewidth=0.8)
ax.set(title='Same query: LIME approximation for malignant probability',
       xlabel='Local surrogate weight; positive supports malignant')
plt.tight_layout()
plt.show()

## PDP and ICE: average and individual response curves

For the two highest-MDI features, partial dependence replaces a feature value across held-out rows and averages the forest's **malignant probability**. Individual conditional expectation (ICE) shows how responses vary across individual rows; 30 curves are displayed for readability. The thick average still uses every held-out row.

These are response curves for the fixed model, not refitted models. They can evaluate combinations unlike the observed data when features are correlated. The next section uses the same features and probability output for ALE, which calculates changes within local data intervals.

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

curve_features = mdi_table.head(2)['feature'].tolist()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
pdp_display = PartialDependenceDisplay.from_estimator(
    model, X_test, features=curve_features, kind='both', method='brute',
    response_method='predict_proba', target=explained_class,
    subsample=min(30, len(X_test)), grid_resolution=20,
    random_state=42, n_jobs=2, ax=axes,
)
for result in pdp_display.pd_results:
    assert np.isfinite(result['average']).all()
    assert np.isfinite(result['individual']).all()
for ax in axes:
    ax.set_ylabel('Predicted P(malignant)')
    ax.set_ylim(-0.02, 1.02)
fig.suptitle('Original forest: PDP averages and ICE curves')
plt.tight_layout()
plt.show()

## ALE: accumulated local effects on the same probability

ALE changes one feature between the edges of small intervals containing observed rows, then accumulates and centers those changes. Using the same held-out rows as PDP makes the reference sample explicit. Centering means the vertical axis is a **relative probability effect**, not a predicted probability; zero is not a decision boundary.

ALE reduces the extrapolation problem associated with marginal PDP, but correlated features, finite samples, and bin choices still affect the result. The curve is not a causal effect or an explanation of just the selected query.

In [ ]:
from PyALE import ale

class MalignantProbabilityModel:
    # PyALE expects predict() to return a numeric regression response.
    # Expose the classifier's probability, rather than its 0/1 labels.
    def predict(self, rows):
        return forest_probabilities(rows)[:, positive_class_index]

ale_results = {}
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for feature, ax in zip(curve_features, axes):
    effect = ale(
        X=X_test, model=MalignantProbabilityModel(), feature=[feature],
        feature_type='continuous', grid_size=15, include_CI=False, plot=False,
    )
    assert len(effect) > 1 and np.isfinite(effect['eff']).all()
    ale_results[feature] = effect
    ax.plot(effect.index, effect['eff'], marker='o', markersize=3, color='#318c72')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set(title=feature, xlabel=feature, ylabel='Centered effect on P(malignant)')
fig.suptitle('Original forest: accumulated local effects')
plt.tight_layout()
plt.show()

## Global surrogate: a small tree approximating the forest

A depth-3 regression tree is trained on the forest's **training-set malignant probabilities**, not on the true diagnoses. Its held-out MAE and R² measure fidelity to the forest. Threshold agreement measures whether the two probability outputs fall on the same side of 0.5. These are fidelity measures, not accuracy against the clinical labels.

The plotted rules belong to a separate approximation. The final lines trace the same selected query through that surrogate and show its probability beside the forest's. A readable tree can still miss behavior of the full forest.

In [ ]:
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.metrics import mean_absolute_error, r2_score

surrogate_tree = DecisionTreeRegressor(max_depth=3, min_samples_leaf=12, random_state=42)
surrogate_tree.fit(X_train, model.predict_proba(X_train)[:, positive_class_index])
surrogate_probability = surrogate_tree.predict(X_test)
assert surrogate_probability.shape == (len(X_test),)
assert np.isfinite(surrogate_probability).all()
surrogate_fidelity = pd.DataFrame([{
    'held-out probability MAE': mean_absolute_error(reference_test_probability, surrogate_probability),
    'held-out probability R²': r2_score(reference_test_probability, surrogate_probability),
    'agreement at threshold 0.5': np.mean((surrogate_probability >= 0.5) == (reference_test_probability >= 0.5)),
}])
display(surrogate_fidelity)

fig, ax = plt.subplots(figsize=(18, 6))
plot_tree(surrogate_tree, feature_names=feature_names, filled=True,
          rounded=True, precision=3, fontsize=8, ax=ax)
ax.set_title('Global surrogate; each leaf value estimates the forest P(malignant)')
plt.tight_layout()
plt.show()

node = 0
query_rule = []
while surrogate_tree.tree_.children_left[node] != surrogate_tree.tree_.children_right[node]:
    feature_index = surrogate_tree.tree_.feature[node]
    threshold = surrogate_tree.tree_.threshold[node]
    goes_left = query_values[feature_index] <= threshold
    operator = '<=' if goes_left else '>'
    query_rule.append(f'{feature_names[feature_index]} {operator} {threshold:.4g}')
    node = (surrogate_tree.tree_.children_left[node] if goes_left
            else surrogate_tree.tree_.children_right[node])
print('Same-query surrogate rule: ' + ' AND '.join(query_rule))
print(f'Forest P(malignant): {forest_query_probability:.4f}')
print(f'Surrogate P(malignant): {surrogate_tree.predict(query)[0]:.4f}')

## Anchors: a sufficient local rule for this prediction

An Anchor is a short conjunction of feature conditions under which the **same random forest** usually keeps its predicted class. The search below uses training-data quartiles, at most three conditions, and a requested precision of 90%. The displayed precision and coverage are estimates under Anchor's perturbation distribution; they are not test accuracy or guarantees for real patients. A bounded search can return a rule below the requested precision, which we report explicitly. Unlike SHAP or LIME, an Anchor does not assign an additive contribution to every feature.

In [ ]:
from anchor import anchor_tabular

# Anchor generates NumPy rows; preserve the reference model's feature schema.
def anchor_predict(rows):
    return model.predict(pd.DataFrame(rows, columns=feature_names))

np.random.seed(42)
anchor_explainer = anchor_tabular.AnchorTabularExplainer(
    class_names=class_names,
    feature_names=feature_names,
    train_data=X_train.to_numpy(),
    discretizer="quartile",
)
anchor_threshold = 0.90
anchor_explanation = anchor_explainer.explain_instance(
    query_values, anchor_predict,
    threshold=anchor_threshold,
    delta=0.10, tau=0.10,
    batch_size=100, max_anchor_size=3,
    coverage_samples=2000,
)
anchor_summary = pd.DataFrame([{
    "explained_prediction": class_names[int(model.predict(query)[0])],
    "rule": " AND ".join(anchor_explanation.names()) or "(empty rule)",
    "estimated_precision": anchor_explanation.precision(),
    "estimated_coverage": anchor_explanation.coverage(),
    "requested_precision": anchor_threshold,
    "threshold_met": anchor_explanation.precision() >= anchor_threshold,
}])
with pd.option_context("display.max_colwidth", None):
    display(anchor_summary.T.rename(columns={0: "Anchor result"}))

## DiCE: observed examples with the opposite model prediction

DiCE's KD-tree method searches **training rows** classified differently by the same forest. We disable its later feature-adjustment step so the results remain observed feature combinations, rather than arbitrary mixtures of 30 correlated measurements. The model's opposite prediction defines success; the original diagnostic labels of these neighboring examples are not used as the search target.

These are model counterfactuals, not causal interventions or advice about changing a diagnosis. They answer “which nearby observed feature profiles receive another model prediction?” They do not show that changing a measurement would change a patient's condition. Distances depend on the search's feature scaling.

In [ ]:
import dice_ml

# The data object describes the training distribution; the model supplies predictions.
dice_training = X_train.copy()
dice_training["diagnosis"] = y_train
dice_data = dice_ml.Data(
    dataframe=dice_training,
    continuous_features=list(feature_names),
    outcome_name="diagnosis",
    # Avoid mode-based rounding (e.g., repeated zeros) erasing small measurements.
    continuous_features_precision={name: 12 for name in feature_names},
)
dice_model = dice_ml.Model(model=model, backend="sklearn")
dice_explainer = dice_ml.Dice(dice_data, dice_model, method="kdtree")
dice_result = dice_explainer.generate_counterfactuals(
    query,
    total_CFs=3,
    desired_class="opposite",
    posthoc_sparsity_param=0,
    verbose=False,
)
cf_features = dice_result.cf_examples_list[0].final_cfs_df.loc[:, feature_names].astype(float)
cf_predictions = model.predict(cf_features)
assert len(cf_features) > 0, "DiCE did not return a counterfactual."
assert np.all(cf_predictions != model.predict(query)[0]), "A returned row did not flip the model prediction."

# Verify the observed-row interpretation against the original training matrix.
matching_rows = np.isclose(
    X_train.to_numpy()[:, None, :], cf_features.to_numpy()[None, :, :],
    rtol=1e-6, atol=1e-8,
).all(axis=2)
assert matching_rows.any(axis=0).all(), "A counterfactual no longer matches an observed training row."
cf_source_positions = matching_rows.argmax(axis=0)
cf_labels = [f"CF {i + 1}" for i in range(len(cf_features))]
cf_summary = pd.DataFrame({
    "source_training_index": X_train.index.to_numpy()[cf_source_positions],
    "forest_prediction": [class_names[int(label)] for label in cf_predictions],
    "forest_P_malignant": model.predict_proba(cf_features)[:, explained_class],
}, index=cf_labels)
display(cf_summary)

# Show the eight largest changes measured in training-standard-deviation units.
# These units are for this display, not DiCE's internal distance metric.
cf_changes = cf_features.subtract(query.iloc[0], axis="columns")
training_scales = X_train.std().replace(0, 1)
largest_changes = cf_changes.abs().div(training_scales, axis="columns").max().nlargest(8).index
cf_comparison = pd.concat([
    query.reset_index(drop=True), cf_features.reset_index(drop=True),
], ignore_index=True)
cf_comparison.index = ["Query"] + cf_labels
display(cf_comparison.loc[:, largest_changes].T)

## RuleFit: a global rule surrogate for the forest

RuleFit combines learned rules and linear feature terms. Here it learns the **forest's training predictions**, rather than the diagnostic labels, so it serves as a global surrogate for our reference model. Its rules summarize the surrogate; they are not rules extracted directly from the forest.

Held-out **fidelity** measures agreement with the forest. Held-out **accuracy** measures agreement with the original diagnostic labels. These quantities answer different questions and are reported separately. We also compare both models on the same query. The rule coefficients belong to the surrogate's logistic model and are not additive forest-probability attributions.

This section requires the installed `h2o` package and Java 17 or 21. It starts a dedicated local H2O server with two threads and a 2 GB heap, then shuts down that server after collecting the results.

In [ ]:
import h2o
from h2o.backend import H2OLocalServer
from h2o.estimators import H2ORuleFitEstimator
from sklearn.metrics import accuracy_score

# Start an owned server instead of attaching to an unrelated existing cluster.
rulefit_server = H2OLocalServer.start(
    nthreads=2, max_mem_size=2 * 1024**3,
    port="54321+", bind_to_localhost=True, verbose=False,
)
try:
    h2o.connect(server=rulefit_server, verbose=False)
    h2o.no_progress()
    surrogate_training = X_train.copy()
    surrogate_training["forest_label"] = model.predict(X_train).astype(str)
    h2o_training = h2o.H2OFrame(surrogate_training)
    h2o_training["forest_label"] = h2o_training["forest_label"].asfactor()
    rulefit_surrogate = H2ORuleFitEstimator(
        min_rule_length=2, max_rule_length=3,
        max_num_rules=20, rule_generation_ntrees=30,
        model_type="rules_and_linear", seed=42,
    )
    rulefit_surrogate.train(
        x=list(feature_names), y="forest_label", training_frame=h2o_training,
    )
    rulefit_test = rulefit_surrogate.predict(h2o.H2OFrame(X_test)).as_data_frame()
    rulefit_predictions = rulefit_test["predict"].astype(int).to_numpy()
    rulefit_query = rulefit_surrogate.predict(h2o.H2OFrame(query)).as_data_frame()
    rulefit_rules = rulefit_surrogate.rule_importance().as_data_frame()
    rulefit_summary = pd.DataFrame({
        "metric": [
            "Forest held-out accuracy vs diagnostic labels",
            "RuleFit held-out fidelity vs forest predictions",
            "RuleFit held-out accuracy vs diagnostic labels",
        ],
        "value": [
            accuracy_score(y_test, model.predict(X_test)),
            accuracy_score(model.predict(X_test), rulefit_predictions),
            accuracy_score(y_test, rulefit_predictions),
        ],
    })
    rulefit_query_comparison = pd.DataFrame({
        "model": ["Reference forest", "RuleFit surrogate"],
        "predicted_class": [
            class_names[int(model.predict(query)[0])],
            class_names[int(rulefit_query.loc[0, "predict"])],
        ],
        "P_malignant": [
            model.predict_proba(query)[0, explained_class],
            float(rulefit_query.loc[0, "p1"]),
        ],
    })
finally:
    # Stop only the server created by this cell, including on an exception.
    rulefit_server.shutdown()

display(rulefit_summary)
display(rulefit_query_comparison)
with pd.option_context("display.max_colwidth", None, "display.max_columns", None):
    display(rulefit_rules.head(10))

## MERLIN: compare two versions on the same cases

MERLIN asks a different question: which rules were added, removed, or preserved between two models? Here a shallow forest represents an earlier version, and the reference forest represents the updated version. Both learn from the same training split and are evaluated on the same test set.

MERLIN 0.1.2 has limitations in its continuous-threshold implementation. Its **explanatory surrogates** therefore use binary concepts, such as “mean radius is above its training-set median.” The original classifiers still receive all continuous measurements. This coarser view can lose fidelity; the table below measures that loss on held-out cases. The median thresholds are descriptive, not clinical cutoffs.

In [ ]:
def compare_merlin_versions(X_train, X_test, previous_model, reference_model,
                            output_dir="comparison_merlin_results"):
    import logging
    from pathlib import Path

    import numpy as np
    import pandas as pd
    from merlin import MERLIN
    from pyeda.inter import expr

    if not X_train.columns.equals(X_test.columns):
        raise ValueError("Training and test features must use the same columns.")
    medians = X_train.median()
    concepts_train = X_train.gt(medians).astype(int).reset_index(drop=True)
    concepts_test = X_test.gt(medians).astype(int).reset_index(drop=True)
    # Simple unique identifiers avoid MERLIN/PyEDA variable-name restrictions.
    names = [f"f{i:02d}_above_median" for i in range(X_train.shape[1])]
    concepts_train.columns = names
    concepts_test.columns = names
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    model_versions = {"left": previous_model, "right": reference_model}
    train_predictions = {
        side: model.predict(X_train).astype(str)
        for side, model in model_versions.items()
    }
    expected_classes = set(np.unique(train_predictions["left"]))
    if expected_classes != set(np.unique(train_predictions["right"])):
        raise ValueError("Both model versions must predict the same class set.")

    comparison = MERLIN(
        concepts_train, train_predictions["left"],
        concepts_train, train_predictions["right"],
        data_type="tabular", surrogate_type="sklearn",
        hyperparameters={"max_depth": 3, "min_samples_leaf": 0.03},
        hyperparameters_selection=False, surrogate_test_size=0.25,
        save_path=str(output_dir), log_level=logging.WARNING,
        save_surrogates=False, save_bdds=False, save_csvs=False,
    )
    comparison.run_trace()
    trace = comparison.trace_results
    # Upstream catches some trace exceptions; require all expected results.
    expected = {(side, label) for side in model_versions for label in expected_classes}
    observed = set(zip(trace["time_label"], trace["class_id"].astype(str)))
    if observed != expected:
        raise RuntimeError("MERLIN did not produce every expected surrogate.")
    comparison.run_explain()
    if set(comparison.explain_results) != expected_classes:
        raise RuntimeError("MERLIN did not explain every expected class.")

    # MERLIN normalizes its variable names. Use the actual names for evaluation.
    concepts_test.columns = comparison.data_manager["left"].X.columns
    concept_legend = pd.DataFrame({
        "concept": concepts_test.columns,
        "original_feature": X_train.columns,
        "training_median": medians.to_numpy(),
        "meaning": "1 means the feature exceeds its training median",
    })
    test_predictions = {
        side: model.predict(X_test).astype(str)
        for side, model in model_versions.items()
    }
    heldout = []
    for row in trace.itertuples(index=False):
        formula = expr(row.bdd_string)
        variables = formula.inputs
        rule_predictions = np.array([
            formula.restrict({variable: int(record[str(variable)])
                              for variable in variables}).is_one()
            for record in concepts_test.to_dict("records")
        ])
        target = test_predictions[row.time_label] == str(row.class_id)
        heldout.append({
            "version": "previous" if row.time_label == "left" else "reference",
            "class_id": str(row.class_id),
            "test_rule_agreement": np.mean(rule_predictions == target),
            "internal_balanced_surrogate_accuracy": row.balanced_accuracy,
        })
    return {
        "explainer": comparison,
        "concept_legend": concept_legend,
        "heldout_fidelity": pd.DataFrame(heldout),
        "rule_changes": pd.DataFrame.from_dict(comparison.explain_results, orient="index"),
    }


In [ ]:
previous_model = RandomForestClassifier(
    n_estimators=30, max_depth=2, min_samples_leaf=4,
    random_state=42, n_jobs=2
).fit(X_train, y_train)
print(f"Previous forest test accuracy: {previous_model.score(X_test, y_test):.3f}")
print(f"Reference forest test accuracy: {model.score(X_test, y_test):.3f}")
print(f"Test prediction disagreement: {np.mean(previous_model.predict(X_test) != predictions):.3f}")
merlin_comparison = compare_merlin_versions(
    X_train, X_test, previous_model, model,
    output_dir="comparison_merlin_results"
)
display(merlin_comparison["heldout_fidelity"])
display(merlin_comparison["concept_legend"])
print("Rule-change summary by class (BDD paths, not patient counts):")
display(merlin_comparison["rule_changes"][["sat_add", "sat_del", "sat_still", "add", "del", "still"]].rename(columns={
    "sat_add": "Added paths", "sat_del": "Removed paths", "sat_still": "Retained paths",
    "add": "Added share", "del": "Removed share", "still": "Retained share",
}))

## Integrated Gradients: the same case, a different model

A random forest has no useful input gradients. Train a small neural network on the **same split**, and report its own held-out performance. Standardization is fitted on training rows only. We still explain the malignant output for the same patient, even if the neural model and forest disagree.

The baseline is the training-set mean (zero after standardization). Attributions sum approximately to the neural model's change in the malignant **logit**, not to the forest's probability. Their raw magnitudes should not be ranked against forest SHAP/LIME values.

In [ ]:
import tensorflow as tf
from sklearn.preprocessing import StandardScaler

tf.keras.utils.set_random_seed(42)
scaler = StandardScaler().fit(X_train)
X_train_nn = scaler.transform(X_train).astype(np.float32)
X_test_nn = scaler.transform(X_test).astype(np.float32)
query_nn = scaler.transform(query).astype(np.float32)[0]
neural_model = tf.keras.Sequential([
    tf.keras.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(2),
])
neural_model.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
)
neural_model.fit(X_train_nn, y_train, epochs=50, batch_size=32, verbose=0)
neural_probabilities = tf.nn.softmax(neural_model(X_test_nn, training=False), axis=1).numpy()[:, 1]
neural_query_probability = float(tf.nn.softmax(neural_model(query_nn[None], training=False), axis=1)[0, 1])
display(pd.DataFrame([{
    "Model": "Tabular neural network (IG)",
    "Test accuracy": accuracy_score(y_test, neural_probabilities >= 0.5),
    "Test ROC AUC": roc_auc_score(y_test, neural_probabilities),
    "Shared-case P(malignant)": neural_query_probability,
    "Agreement with forest on test": accuracy_score(predictions, neural_probabilities >= 0.5),
}]))

In [ ]:
def integrated_gradients(model, input_sample, baseline, num_steps=50, target_class=None):
    """Attribute one fixed class output along a baseline-to-input path.

    Inputs are one-dimensional feature vectors. By default, explain the class
    predicted at the actual input. Trapezoidal integration uses both endpoints.
    """
    if num_steps < 1:
        raise ValueError("num_steps must be at least 1")
    input_sample = tf.convert_to_tensor(input_sample, dtype=tf.float32)
    baseline = tf.convert_to_tensor(baseline, dtype=tf.float32)
    if input_sample.shape.rank != 1 or input_sample.shape != baseline.shape:
        raise ValueError("input_sample and baseline must be matching feature vectors")

    if target_class is None:
        target_class = int(tf.argmax(model(input_sample[None, :], training=False)[0]))

    alphas = tf.linspace(0.0, 1.0, num_steps + 1)
    path = baseline[None, :] + alphas[:, None] * (input_sample - baseline)[None, :]
    with tf.GradientTape() as tape:
        tape.watch(path)
        # Keep the target class fixed, even if the prediction changes on the path.
        target_scores = model(path, training=False)[:, target_class]
    gradients = tape.gradient(target_scores, path)
    trapezoids = (gradients[:-1] + gradients[1:]) / 2.0
    average_gradient = tf.reduce_mean(trapezoids, axis=0)
    return ((input_sample - baseline) * average_gradient).numpy()

In [ ]:
ig_values = integrated_gradients(
    neural_model, query_nn, np.zeros_like(query_nn),
    num_steps=256, target_class=explained_class
)
assert ig_values.shape == (len(feature_names),) and np.isfinite(ig_values).all()
ig_logit_change = float(
    neural_model(query_nn[None], training=False)[0, explained_class]
    - neural_model(np.zeros_like(query_nn)[None], training=False)[0, explained_class]
)
print(f"Malignant-logit change from baseline: {ig_logit_change:.5f}")
print(f"Sum of IG attributions: {ig_values.sum():.5f}")
print(f"Numerical completeness residual: {ig_values.sum() - ig_logit_change:.5f}")
ig_series = pd.Series(ig_values, index=feature_names)
ig_top = ig_series.loc[ig_series.abs().nlargest(10).index].sort_values()
ig_top.plot.barh(color=np.where(ig_top >= 0, "firebrick", "steelblue"), figsize=(9, 5))
plt.axvline(0, color="black", linewidth=0.7)
plt.xlabel("Contribution to neural malignant logit relative to training-mean baseline")
plt.title("Integrated Gradients — same patient, tabular neural model")
plt.tight_layout()
plt.show()

## Compare the outputs, not just the pictures

The global ranking table compares three definitions of importance for the **same forest**, with held-out rows used for permutation and SHAP. Rank disagreement is expected: impurity uses training splits, permutation measures held-out performance loss, and SHAP measures average attribution magnitude. Correlated measurements can share or substitute for one another.

The local table places SHAP contributions next to LIME's selected discretized feature conditions for the **same case and malignant output**. A missing LIME value means the ten-feature surrogate did not select that feature; it does not prove that feature is irrelevant. LIME weights do not share SHAP's additive decomposition of the full forest.

In [ ]:
global_ranks = pd.DataFrame(index=feature_names)
for column, table, feature_column, value_column in [
    ("MDI rank", mdi_table, "feature", "MDI"),
    ("Permutation AUC rank", permutation_table, "feature", "AUC decrease"),
    ("Mean absolute SHAP rank", shap_global_table, "feature", "mean absolute SHAP"),
]:
    values = table.set_index(feature_column)[value_column]
    global_ranks[column] = values.rank(ascending=False, method="min")
print("Global rankings for the ten largest mean absolute SHAP features:")
display(global_ranks.loc[shap_global_table.head(10)["feature"]])

local_comparison = pd.DataFrame({
    "feature": feature_names,
    "SHAP probability contribution": local_shap.values,
}).set_index("feature")
lime_index_weights = dict(lime_query.as_map()[explained_class])
lime_condition_names = dict(zip(
    [index for index, _ in lime_query.as_map()[explained_class]],
    [condition for condition, _ in lime_query.as_list(label=explained_class)],
))
local_comparison["LIME weight"] = [lime_index_weights.get(i, np.nan) for i in range(len(feature_names))]
local_comparison["LIME feature condition"] = [lime_condition_names.get(i, "not selected") for i in range(len(feature_names))]
local_order = local_comparison["SHAP probability contribution"].abs().sort_values(ascending=False).index
print("Local contributions for the shared case (ordered by absolute SHAP value):")
display(local_comparison.loc[local_order].head(15))

### Diagnostics with their own meanings

These are checks on different explanatory tasks, not interchangeable scores. For example, Anchor precision describes agreement under its perturbation distribution; surrogate fidelity describes approximation of another model; neither is diagnostic accuracy.

In [ ]:
diagnostics = pd.DataFrame([
    ("SHAP", "Absolute additive reconstruction error", abs(float(local_shap.base_values + local_shap.values.sum()) - forest_query_probability), "probability"),
    ("LIME", "Absolute shared-case approximation error", abs(lime_local_prediction - forest_query_probability), "probability"),
    ("LIME", "Weighted local surrogate R²", lime_query.score, "local R²"),
    ("Shallow surrogate", "Held-out probability MAE", surrogate_fidelity.loc[0, "held-out probability MAE"], "probability"),
    ("Anchors", "Estimated conditional prediction agreement", anchor_explanation.precision(), "fraction"),
    ("Anchors", "Estimated coverage", anchor_explanation.coverage(), "fraction"),
    ("DiCE", "Verified opposite-prediction observed rows", len(cf_features), "rows"),
    ("RuleFit surrogate", "Held-out agreement with forest", accuracy_score(predictions, rulefit_predictions), "fraction"),
    ("MERLIN", "Lowest held-out class-rule agreement", merlin_comparison["heldout_fidelity"]["test_rule_agreement"].min(), "fraction"),
    ("Integrated Gradients", "Absolute numerical completeness residual", abs(float(ig_values.sum()) - ig_logit_change), "neural logit"),
], columns=["Method", "Diagnostic", "Value", "Units"])
assert np.isfinite(diagnostics["Value"]).all()
display(diagnostics)

## What to take away

- Use **permutation importance** to ask about held-out predictive performance, and inspect correlation before interpreting a low score as irrelevance. **MDI** and **drop-column** answer different training/refitting questions.
- Use **SHAP or LIME** for this forest prediction, and inspect LIME's local approximation error before trusting its simplified story.
- Use **PDP/ICE and ALE** to explore response patterns. PDP can evaluate unrealistic combinations of correlated measurements; ALE reduces that extrapolation but is still not causal.
- Use **Anchors** for a sampled local rule and **DiCE** for alternative model decisions. Observed counterfactual neighbors are examples, not feasible changes for the same person.
- Use **shallow trees or RuleFit** for a simpler global approximation, while reporting its measured fidelity. Use **MERLIN** to compare rule changes between model versions and inspect the fidelity of its binary concept view.
- Use **Integrated Gradients** when explaining the neural network itself, with a stated baseline and completeness check. It does not explain the forest.

**Try next:** choose a different test row, change the random seed, or vary the model's depth. Keep the split and case fixed across methods, and see which conclusions persist. A persuasive plot is not a substitute for a faithful explanation.